So for each input $x$, we will predict a probability distribution function $P(y|x)$:

$P(y|x) = \sum_{k}^{K} \Pi_{k}(x) \phi(y, \mu_{k}(x), \sigma_{k}(x))$

- $k$ is an index describing which Gaussian we are referencing. There are $K$ Gaussians total.
- $\sum_{k}^{K}$ is the summation operator. We sum every $k$ Gaussian across all $K$. You might also see $\sum_{k=0}^{K-1}$ or $\sum_{k=1}^{K}$ depending on whether an author is using zero-based numbering or not.
- $\Pi_k$ acts as a weight, or multiplier, for mixing every $k$ Gaussian. It is a function of the input $x$: $\Pi_k(x)$
- $\phi$ is the Gaussian function and returns the at $y$ for a given mean and standard deviation.
- $\mu_k$ and $\sigma_k$ are the parameters for the $k$ Gaussian: mean $\mu_k$ and standard deviation $\sigma_k$. Instead of being fixed for each Gaussian, they are also functions of the input $x$: $\mu_k(x)$ and $\sigma_k(x)$

We use a [*softmax*](https://en.wikipedia.org/wiki/Softmax_function) operator to ensure that $\Pi$ sums to one across all $k$, and the exponential function ensures that each weight $\Pi_k$ is positive. We also use the exponential function to ensure that every $\sigma_k$ is positive.

In [ ]:
import torch.nn as nn

class MDN(nn.Module):
    def __init__(self, n_hidden, n_gaussians):
        super(MDN, self).__init__()

        # Generate a latent z from the input data
        # (i think, this is equivalent to our autoencoder)
        self.z_h = nn.Sequential(
            nn.Linear(1, n_hidden),
            nn.Tanh()
        )

        # Parameters for Mixture of Gaussians 
        self.z_pi = nn.Linear(n_hidden, n_gaussians) # weight for each gaussian
        self.z_sigma = nn.Linear(n_hidden, n_gaussians)
        self.z_mu = (nn.Linear(n_hidden, n_gaussians))
    
    def forward(self, x):
        z_h = self.z_h(x)
        pi = nn.functional.softmax(self.z_pi(z_h), -1)
        sigma = torch.exp(self.z_sigma(z_h))
        mu = self.z_mu(z_h)
        return pi, sigma, mu


We cannot use the `MSELoss()` function for this task, because the output is an entire description of the probability distribution and not just a single value. A more suitable loss function is the logarithm of the likelihood of the output distribution vs the training data:

$loss(y | x) = -\log[ \sum_{k}^{K} \Pi_{k}(x) \phi(y, \mu_{k}(x), \sigma_{k}(x)) ]$

So for every $x$ input and $y$ output pair in the training data set, we can compute a loss based on the predicted distribution versus the actual distribution, and then attempt the minimise the sum of all the costs combined. To those who are familiar with logistic regression and cross entropy minimisation of softmax, this is a similar approach, but with non-discretised states.

We have to implement this cost function ourselves:

In [ ]:
import numpy as np
oneDivSqrtTwoPI = 1.0 / np.sqrt(2.0*np.pi) # normalization factor for Gaussians
def gaussian_distribution(y, mu, sigma):
    # make mu=K copies of y, subtruct mu, divide by sigma
    result = (y.expand_as(mu) - mu) * torch.reciprocal(sigma)
    result = -0.5 * (result * result)
    return (torch.exp(result) * torch.reciprocal(sigma)) * oneDivSqrtTwoPI

def mdn_loss_fn(pi, sigma, mu, y):
    result = gaussian_distribution(y, mu, sigma) * pi
    result = torch.sum(result, dim=1)
    result = -torch.log(result)
    return torch.mean(result)

In [ ]:
import torch 
from torch.autograd import Variable # storing data while learning

network = MDN(n_hidden=20, n_gaussians=5)
optimizer = torch.optim.Adam(network.parameters())

# data
def generate_data(n_samples):
    epsilon = np.random.normal(size=(n_samples))
    x_data = np.random.uniform(-10.5, 10.5, n_samples)
    y_data = 7*np.sin(0.75*x_data) + 0.5*x_data + epsilon
    return x_data, y_data
    
n_samples = 1000
n_input = 1
n_hidden = 20
n_output = 1
x_data, y_data = generate_data(n_samples)
x_tensor = torch.from_numpy(np.float32(x_data).reshape(n_samples, n_input))
y_tensor = torch.from_numpy(np.float32(y_data).reshape(n_samples, n_input))
mdn_x_data = y_data
mdn_y_data = x_data

mdn_x_tensor = y_tensor
mdn_y_tensor = x_tensor

x_variable = Variable(mdn_x_tensor)
y_variable = Variable(mdn_y_tensor, requires_grad=False)

In [ ]:
def train_mdn():
    for epoch in range(10000):
        pi_variable, sigma_variable, mu_variable = network(x_variable)
        loss = mdn_loss_fn(pi_variable, sigma_variable, mu_variable, y_variable)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if epoch % 500 == 0:
            print(epoch, loss.item())

train_mdn()

# MDN-RNN Architecture & Loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MDN_RNN(nn.Module):
    def __init__(self, latent_dim, action_dim, hidden_size, num_layers, n_gaussians):
        super().__init__()
        self.latent_dim = latent_dim
        self.action_dim = action_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.n_gaussians = n_gaussians

        # Input: (N, L, latent_dim + action_dim)
        self.lstm = nn.LSTM(latent_dim + action_dim, 
                            hidden_size, 
                            num_layers, 
                            batch_first=True)
        
        self.pi = nn.Linear(hidden_size, n_gaussians)
        self.mu = nn.Linear(hidden_size, n_gaussians * latent_dim)
        self.sigma = nn.Linear(hidden_size, n_gaussians * latent_dim)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)      # (N, L, H_out)
        logits_pi = self.pi(lstm_out)   # (N, L, n_gaussians)
        pi = F.softmax(logits_pi, -1)   # (N, L, n_g)
        
        batch_dim, seq_dim, _ = lstm_out.shape
        
        mu = self.mu(lstm_out)                                                                      # (N, L, n_g * l_dim)
        mu = mu.view(batch_dim, seq_dim, self.n_gaussians, self.latent_dim)                         # (N, L, n_g, l_dim)

        sigma = self.sigma(lstm_out)                                                                # (N, L, n_g * l_dim)
        sigma = torch.exp(sigma.view(batch_dim, seq_dim, self.n_gaussians, self.latent_dim)) + 1e-3 # (N, L, n_g, l_dim)

        return pi, mu, sigma
    
def gaussian_density(mu, sigma, y):
    """
    Compute per-component, per-timestep multivariate Gaussian density

    mu: [N, L, n_g, l_dim]
    sigma: [N, L, n_g, l_dim]
    y: [N, L, l_dim]

    returns (N, L, n_g) densities 

    """
    norm = 1.0 / torch.sqrt(torch.tensor(2.0) * torch.pi) 
    # y.shape     = [N, L, latent_dim] -> [N, L, 1, l_dim]
    y = y.unsqueeze(2)
    # mu.shape    = [N, L, n_g, l_dim]
    # sigma.shape = [N, L, n_g, l_dim]
    per_dim = torch.exp(-0.5 * ((y-mu)/sigma)**2) / (sigma) * norm # [N, L, n_g, l_dim]

    # product over l_dim to get the full latent dimension density
    # p(y | m_k, s_k)
    return torch.prod(per_dim, dim=-1) # (N, L, n_g)

def mdn_loss(pi, mu, sigma, y): 
    """
    pi: [N, L, n_g]
    """
    res = pi * gaussian_density(mu, sigma, y) # [N, L, n_g]
    res = torch.sum(res, -1) # P(y) - [N, L]
    res = -torch.log(res) # -log(P(y)) # [N, L]
    return torch.mean(res)

# Training

### Pre-compute Z-latent dataset with VAE

In [ ]:
import sys
sys.path.append("..")
from torch.utils.data import DataLoader
import torch
from data.dataset import CarRacingDataset
from modules.vae import VAE, vae_loss
import numpy as np
import os

def precompute_latents(vae_path, data_dir, processed_dir):
    # load the model
    vae = VAE(32)
    vae.load_state_dict(torch.load(vae_path, map_location='cpu'))
    vae.eval()

    # load dataset
    data = CarRacingDataset(data_dir)
    dl = DataLoader(data, batch_size=16)
    # Process each batch and accumulate latents by sequence
    sequence_data = {}  # Dictionary to store data by file_idx (sequence)
    
    for batch_frames, batch_actions, batch_file_idxs in dl:
        # Forward pass through VAE
        recon_x, mu, log_var, z = vae(batch_frames)  # z: (batch_size, 32)
        
        # Group by sequence (file_idx)
        for i in range(len(batch_file_idxs)):
            file_idx = batch_file_idxs[i].item()
            
            if file_idx not in sequence_data:
                sequence_data[file_idx] = {
                    'latents': [],
                    'actions': []
                }
            
            sequence_data[file_idx]['latents'].append(z[i].detach().numpy())
            sequence_data[file_idx]['actions'].append(batch_actions[i].numpy())
    
    # Save each sequence as a separate file
    os.makedirs(processed_dir, exist_ok=True)
    for seq_idx, seq_data in sequence_data.items():
        data = {
            'latent_observations': np.array(seq_data['latents']),
            'actions': np.array(seq_data['actions'])
        }
        
        np.savez_compressed(
            os.path.join(processed_dir, f'rollout_{seq_idx:05d}.npz'),
            **data
        )
    
precompute_latents('../train/vae_l32_epoch1.pt', '../data/roll100_maxts1k', '../data/latent_roll100_maxts1k')
# precompute_latents('../train/vae_l32_epoch1.pt', '../data/test_rollouts', '../data/test_latents')


### Potential Issue: The difference between sequential images is too little. Skip frames?

Skipping frames might result in different issues like bad modeling of the physics, won't do for now

### Decode latents to verify

In [ ]:
import sys
sys.path.append("..")
from torch.utils.data import DataLoader
from data.dataset import LatentSequenceDataset
from modules.vae import VAE
import matplotlib.pyplot as plt
import torch
import random

d = LatentSequenceDataset('../data/latent_roll100_maxts1k')
dl = DataLoader(d, 16)


vae = VAE(32)
vae.load_state_dict(torch.load(f'../train/vae_l32_epoch1.pt', map_location='cpu'))
vae.eval()




In [ ]:
for latents, actions, rollout_idxs in dl:
    print(f'len(latents): {len(latents)}')
    print(f'len(actions): {len(actions)}')
    # Reconstruct images from latents and display in a grid
    batch_size = latents.size(0)
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.flatten()
    
    for i, l in enumerate(latents):
        if i >= 16:  # Only show first 16 images
            break
        img = vae.decoder(l)
        img_np = img.detach().squeeze(0).permute(1,2,0).numpy()
        axes[i].imshow(img_np)
        axes[i].set_title(f"Latent {i}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    break  # Only process first batch


### Train

In [ ]:
import sys
sys.path.append('..')
from data.dataset import LatentSequenceDataset
from torch.utils.data import DataLoader

d = LatentSequenceDataset('../data/latent_roll100_maxts1k', sequence_length=500)
dl = DataLoader(d, batch_size=16, shuffle=True)





### Overfit single batch

In [ ]:
model = MDN_RNN(32, 3, 256, 1, 5)
total_params = sum(p.numel() for p in model.parameters())

opt = torch.optim.Adam(model.parameters(), 1e-3)
x_train, y_train = next(iter(dl)) # (B, S, l+a), (B, S, l)
epochs = 10

for epoch in range(epochs):
    model.zero_grad()

    # forward
    pi, mu, sigma = model(x_train)

    # loss
    loss = mdn_loss(pi, mu, sigma, y_train)
    loss.backward() # backward pass
    print(f'Epoch: {epoch}, Loss: {loss:.4f}')

    # update
    opt.step()




### Train

In [ ]:
import wandb

# load data
import sys
sys.path.append('..')
from data.dataset import LatentSequenceDataset
from torch.utils.data import DataLoader, random_split

# fix seed
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed = 99
set_seed(seed)

# load data
batch_size = 16
seq_len = 500
val_split = 0.2

d = LatentSequenceDataset('../data/latent_roll100_maxts1k', sequence_length=seq_len)

val_size = int(len(d)*val_split)
train_size = len(d) - val_size

train_dataset, val_dataset = random_split(
    d,
    [train_size,val_size],
    generator=torch.Generator().manual_seed(seed)
)

# split into train val
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# hyperparams
latent_dim = 32
action_dim = 3
hidden_size = 256
n_layers = 1
n_gaussians = 5
lr = 1e-3
epochs = 1

model = MDN_RNN(latent_dim, action_dim, hidden_size, n_layers, n_gaussians)
opt = torch.optim.Adam(model.parameters(), lr)
print(f'Total params: {sum(p.numel() for p in model.parameters())}')

wandb.login()
wandb.init(
    project="mdn-rnn",
    config={
        "latent_dim": latent_dim,
        "action_dim": action_dim,
        "hidden_size": hidden_size,
        "n_layers": n_layers,
        "n_gaussians": n_gaussians,
        "learning_rate": lr,
        "batch_size": batch_size,
        "sequence_length": seq_len,
        "epochs": epochs,
        "seed": seed,
    }
)
config = wandb.config

# Training loop with validation
def train():
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        opt.zero_grad()
        pi, mu, sigma = model(x)
        loss = mdn_loss(pi, mu, sigma, y)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def validate():
    model.eval()
    val_loss = 0.0
    for x, y in val_loader:
        pi, mu, sigma = model(x)
        val_loss += mdn_loss(pi, mu, sigma, y).item()
    return val_loss / len(val_loader)

# Run training
for epoch in range(epochs):
    train_loss = train()
    val_loss = validate()

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
    })

    print(f'Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}')

# Save model
model_path = f"MDNRNN-e{epochs}.pt"
torch.save(model.state_dict(), model_path)

# Log model to W&B
artifact = wandb.Artifact("mdn_rnn_model", type="model")
artifact.add_file(model_path)
wandb.log_artifact(artifact)
wandb.finish()

### Visualization - Validation